# GPU inference benchmark (Colab)

Fills the one gap this repo's CPU-only benchmark can't: real GPU latency numbers.

**Before running:** `Runtime` -> `Change runtime type` -> select a `T4 GPU` (or any GPU), then `Runtime` -> `Run all`.

This clones the repo, installs Sionna, and times the same refiner networks used in `src/benchmark.py` -- but on GPU, at batch sizes 1 (real-time check), 8, and 256, so the numbers are directly comparable to the CPU numbers already in `results/benchmark.csv` and `results/realtime_latency.csv`.

In [ ]:
!git clone https://github.com/taovietducofficial/Neural-Channel-Estimator-5G.git
%cd Neural-Channel-Estimator-5G
!pip install -q sionna

In [ ]:
import sys, time
import torch
sys.path.insert(0, '.')

from src.models import CNNChannelEstimator, TransformerChannelEstimator
from src.benchmark import TransformerRefiner, grid_shape
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '(no GPU -- check Runtime > Change runtime type)')

In [ ]:
pc = PUSCHConfig()
tx = PUSCHTransmitter(pc)

# Random-initialized weights are fine here: we're only measuring inference
# LATENCY (a function of architecture and shapes), not accuracy, so no
# checkpoint/training is needed for this benchmark.
cnn_full = CNNChannelEstimator(tx.resource_grid, pc).to(device)
transformer_full = TransformerChannelEstimator(tx.resource_grid, pc).to(device)
cnn_refiner = cnn_full.refine
transformer_refiner = TransformerRefiner(transformer_full).to(device)

def bench(model, batch_size, repeats=200, warmup=30):
    x = torch.randn(*grid_shape(batch_size), device=device)
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            model(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return elapsed / repeats * 1000

In [ ]:
print(f"{'batch':>6}{'CNN ms':>12}{'Transformer ms':>18}")
rows = []
for batch in [1, 8, 256]:
    cnn_ms = bench(cnn_refiner, batch)
    trans_ms = bench(transformer_refiner, batch)
    rows.append((batch, cnn_ms, trans_ms))
    print(f"{batch:>6}{cnn_ms:>12.4f}{trans_ms:>18.4f}")

print("\nCopy this table into docs/oran-integration.md or README to replace the")
print("'no GPU numbers' gap noted in Honest limitations.")

with open('results_gpu_benchmark.csv', 'w') as f:
    f.write('batch,cnn_ms,transformer_ms,device\n')
    for batch, cnn_ms, trans_ms in rows:
        f.write(f'{batch},{cnn_ms:.4f},{trans_ms:.4f},{torch.cuda.get_device_name(0) if device=="cuda" else "cpu"}\n')
print('\nsaved results_gpu_benchmark.csv -- download it (folder icon on the left) and paste the numbers back.')